<a href="https://colab.research.google.com/github/lenchanti/Bioinformatics-with-Python-Cookbook-Second-Edition/blob/master/02_Preprocessing_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### *Preprocess Experiment Data (2025.11)(ver2)*

In [ ]:
import sys
from pathlib import Path

# 현재 위치에서 위로 올라가며 utils/params.py를 찾는다
here = Path.cwd()
found = False
for p in [here, *here.parents]:
    if (p / "utils" / "params.py").exists():
        # 두 가지 임포트 방식 모두 지원
        sys.path.insert(0, str(p / "utils"))  # `import params`, `import vggish_input`
        sys.path.insert(0, str(p))            # `from utils import params`도 가능하게
        found = True
        break

if not found:
    raise FileNotFoundError(f"utils/params.py를 {here}부터 상위 폴더에서 찾지 못했습니다.")

# 이제 임포트
import params
import vggish_input


In [ ]:
import os
import sys
from pathlib import Path
import pickle as pkl
import numpy as np
from numpy.lib.stride_tricks import as_strided
from tqdm import tqdm
# from utils import vggish_input, params
import pandas as pd

import sys
from pathlib import Path

# 현재 위치에서 위로 올라가며 utils/params.py를 찾는다
here = Path.cwd()
found = False
for p in [here, *here.parents]:
    if (p / "utils" / "params.py").exists():
        # 두 가지 임포트 방식 모두 지원
        sys.path.insert(0, str(p / "utils"))  # `import params`, `import vggish_input`
        sys.path.insert(0, str(p))            # `from utils import params`도 가능하게
        found = True
        break

if not found:
    raise FileNotFoundError(f"utils/params.py를 {here}부터 상위 폴더에서 찾지 못했습니다.")

import params
import vggish_input
# === 경로/설정 ===
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / ".." / ".." / ".." / "HCAR"))
sys.path.append(str(PROJECT_ROOT / "utils" ))

# Add project directory for HCAR utils
def add_time_zero_row(df, time_col='Time', unixtime_col='UnixTime'):
    """
    Ensure a row at time zero in the DataFrame.
    """
    df = df.copy()
    df[time_col]     = pd.to_numeric(df[time_col],     errors='coerce')
    df[unixtime_col] = pd.to_numeric(df[unixtime_col], errors='coerce')

    if (df[time_col] == 0).any():
        return df

    first = df.iloc[0].copy()
    new = first.copy()
    new[time_col]     = 0
    new[unixtime_col] = first[unixtime_col] - int(first[time_col] * 1000)
    return pd.concat([pd.DataFrame([new]), df], ignore_index=True)

def frame(data, window_length, hop_length):
    """
    Frame 2D array into overlapping windows.
    """
    if data.shape[0] < window_length:
        pad_n = window_length - data.shape[0]
        data = np.vstack([data, np.zeros((pad_n, data.shape[1]))])
    n_frames = 1 + (data.shape[0] - window_length) // hop_length
    shape    = (n_frames, window_length, data.shape[1])
    strides  = (hop_length * data.strides[0],) + data.strides
    return as_strided(data, shape=shape, strides=strides)

NORMALIZE_MAP = {
    'Brushing Teeth': 'Tooth_brushing',
    'Toothbrushing': 'Tooth_brushing',
    'Washing Hands': 'Washing_hands',
    'Vacuum Cleaner': 'Vacuum_Cleaner',
}

def normalize_activity(s: str) -> str:
    if not isinstance(s, str):
        return 'Other'
    s = s.strip()
    if s in NORMALIZE_MAP:
        return NORMALIZE_MAP[s]
    # 공백을 언더스코어로 바꾸되, 우리가 쓰는 클래스가 아니면 Other
    s_us = s.replace(' ', '_')
    allowed = {
        'Tooth_brushing', 'Washing_hands', 'Shower',
        'Wiping', 'Vacuum_Cleaner', 'Other'
    }
    return s_us if s_us in allowed else 'Other'

def rebuild_waveform(df):
    """
    Flatten multi-column audio data to a waveform and return start time.
    """
    cols = [c for c in df.columns if c.startswith('AudioData')]
    arr  = df[cols].to_numpy(dtype=np.int16)
    times= df['UnixTime_s'].to_numpy()
    return arr.flatten(), times[0]

def generate_mel_chunks(waveform, sr=16000, lower_edge_hertz=10,
                         upper_edge_hertz=8000, chunk_secs=10):
    """
    Generate mel spectrogram chunks from a long waveform.
    """
    chunk_samples = int(chunk_secs * sr)
    mel_chunks = []
    for start in tqdm(range(0, len(waveform), chunk_samples)):
        chunk = waveform[start:start + chunk_samples]
        if chunk.size == 0:
            break
        mel = vggish_input.wavform_to_concat_examples(
            chunk,
            lower_edge_hertz=lower_edge_hertz,
            upper_edge_hertz=upper_edge_hertz,
            sr=sr
        )
        mel_chunks.append(mel)
    return np.concatenate(mel_chunks, axis=0)

if __name__ == '__main__':
    DATA_PATH = Path("../../Data/Experiment_Data_ver2/1_RawDataset")
    SAVE_PATH = Path("../../Data/Experiment_Data_ver2/2_PreprocessDataset")

    participants = [d.name for d in DATA_PATH.iterdir() if d.is_dir()]
    class_list = [
        'Tooth_brushing', 'Washing_hands', 'Shower',
        'Wiping', 'Vacuum_Cleaner', 'Other'
    ]

    for participant in participants:
        print(participant)
        # if participant != "302_Day1":
        #     continue

        if "Day" not in participant:
            continue
        # if p in pid_pd:
        #     if p in participant:
        #         print(f"있는 데이터: {pid_pd}")
        #         continue

        folder = DATA_PATH / participant

        # Load CSVs
        anno_df = pd.read_csv(
            next(folder.glob('*annotation.csv')), engine='python', on_bad_lines='skip'
        )
        # pred_df = pd.read_csv(
        #     next(folder.glob('*Predicted_Activity.csv')), engine='python', on_bad_lines='skip'
        # )
        sensor_df = pd.read_csv(
                    next(folder.glob('*SensorData.csv')), engine='python', on_bad_lines='skip'
                )

        anno_df.loc[0,"UnixTime"]=sensor_df.loc[0,"UnixTime"]
        # Harmonize labels
        anno_df["Activity"].str.replace("Brushing Teeth",'Tooth_brushing', regex=False)
        anno_df["Activity"].str.replace("Washing Hands",'Washing_hands', regex=False)
        anno_df["Activity"].str.replace("Vacuum Cleaner",'Vacuum_Cleaner', regex=False)
        anno_df['Activity'] = anno_df['Activity'].apply(normalize_activity)
        # anno_df['Activity'] = anno_df['Activity'].str.replace('Toothbrushing', 'Tooth_brushing', regex=False)
        # pred_df['Predict']   = pred_df['Predict'].str.replace(' ', '_', regex=False)

        # Insert time-zero
        anno_df = add_time_zero_row(anno_df)
        # pred_df = add_time_zero_row(pred_df)
        sensor_df=add_time_zero_row(sensor_df)

        # Align annotation times
        # delta = pred_df.loc[pred_df.Time==0, 'UnixTime'].iloc[0] - \
        #         anno_df.loc[anno_df.Time==0, 'UnixTime'].iloc[0]
        # anno_df['UnixTime'] += delta
        delta = sensor_df.loc[sensor_df.Time==0, 'UnixTime'].iloc[0] - \
                        anno_df.loc[anno_df.Time==0, 'UnixTime'].iloc[0]
        anno_df['UnixTime'] += delta

        # Truncate and add Session Stop
        # end_unix = pred_df['UnixTime'].max()
        end_unix=sensor_df['UnixTime'].max()
        anno_df = anno_df[anno_df['UnixTime'] <= end_unix]
        last = sensor_df.loc[sensor_df.UnixTime.idxmax()].to_dict()

        # last = pred_df.loc[pred_df.UnixTime.idxmax()].to_dict()
        last.update({'Event':'Session Stop','Activity':'','Confirm':''})
        anno_df = pd.concat([anno_df, pd.DataFrame([last])], ignore_index=True)

        # Fix unmatched starts, drop bad ends
        anno_df['Confirm'] = anno_df['Confirm'].fillna('')
        df = anno_df.copy()
        next_act = df['Activity'].shift(-1)
        next_evt = df['Event'].shift(-1)
        df = df[~((df.Event=='Start') & ~((next_act==df.Activity)&(next_evt=='End')))]
        drop = []
        for idx,row in df[(df.Event=='End')&(df.Confirm=='no')].iterrows():
            starts = df[(df.Activity==row.Activity)&(df.Event=='Start')&(df.index<idx)]
            drop += ([starts.index.max(), idx] if not starts.empty else [idx])
        df = df.drop(drop).sort_values(['UnixTime','Time']).reset_index(drop=True)
        anno_df = df
        anno_df=anno_df.iloc[:,:11]

        # Convert to seconds and save CSV
        anno_df['UnixTime_s'] = anno_df['UnixTime'] / 1000.0
        out = SAVE_PATH / participant
        out.mkdir(parents=True, exist_ok=True)
        anno_df.to_csv(out / f"{participant}_Annotation_processed.csv", index=False)
        # pred_df.to_csv(out / f"{participant}_Predicted_Activity_processed.csv", index=False)

    # Process sensor/audio
    for participant in participants:
        print(f"#id={participant}")

        if "Day" not in participant:
            continue
        # if p in pid_pd:
        #     if p in participant:
        #         print(f"있는 데이터: {pid_pd}")
        #         continue

        print(f"#id={participant}")
        folder = DATA_PATH / participant
        sensor_df = pd.read_csv(
            next(folder.glob('*SensorData.csv')), engine='python', on_bad_lines='skip'
        )
        audio_df = pd.read_csv(
            next(folder.glob('*AudioData.csv')), engine='python', on_bad_lines='skip'
        )
        anno_clean = pd.read_csv(
            SAVE_PATH/participant/f"{participant}_Annotation_processed.csv"
        )

        # audio_df=audio_df.drop(columns=["UnixTime_norm"])
        # sensor_df=sensor_df.drop(columns=['UnixTime_norm', 'Time_norm'])

        exp = sensor_df.columns[:17]
        extra = sensor_df.columns[17:]

        # 왼쪽 필수 컬럼에서 결측값이 있는 행
        left_bad = sensor_df[exp].isna().any(axis=1)

        # 오른쪽 추가 컬럼에 값이 있는 행 (있으면 안되는 값들)
        if len(extra) > 0:
            right_bad = sensor_df[extra].notna().any(axis=1)
        else:
            right_bad = pd.Series(False, index=sensor_df.index)

        bad = left_bad | right_bad
        sensor_df = sensor_df[~bad].reset_index(drop=True)

        # Add time-zero rows
        sensor_df = add_time_zero_row(sensor_df)
        audio_df  = add_time_zero_row(audio_df)

        # Sync times
        base_zero = sensor_df.loc[sensor_df.Time==0,'UnixTime'].iloc[0]
        audio_df['UnixTime'] += base_zero - audio_df.loc[audio_df.Time==0,'UnixTime'].iloc[0]
        anno_clean['UnixTime'] += base_zero - anno_clean.loc[anno_clean.Time==0,'UnixTime'].iloc[0]

        # Truncate session
        stop_ts = anno_clean.loc[anno_clean.Event=='Session Stop','UnixTime'].iloc[0]
        sensor_df = sensor_df[sensor_df.UnixTime<=stop_ts].reset_index(drop=True)
        audio_df  = audio_df[audio_df.UnixTime<=stop_ts].reset_index(drop=True)

        # Convert to seconds
        for df in (sensor_df, audio_df, anno_clean):
            df['UnixTime_s'] = df['UnixTime']/1000.0

        # IMU 데이터는 9축만 사용 (논문 Section 4.1.1)
        imu_arr = sensor_df[
            ['AccX','AccY','AccZ','GyroX','GyroY','GyroZ','RotVecX','RotVecY','RotVecZ']
        ].to_numpy()

        # 시간 정보는 별도로 관리
        imu_times = sensor_df['UnixTime_s'].to_numpy()

        iwlen  = int(2.0 * 50)    # 2초 * 50Hz = 100 샘플
        iwstep = int(0.2 * 50)    # 0.2초 * 50Hz = 10 샘플

        # 각각 별도로 프레이밍
        imu_frames = frame(imu_arr, iwlen, iwstep)  # (N, 100, 9)
        time_frames = frame(imu_times.reshape(-1,1), iwlen, iwstep)  # (N, 100, 1)

        print("IMU frames shape:", imu_frames.shape)
        print("Time frames shape:", time_frames.shape)

        # Build intervals
        intervals = []
        stack = {}
        for _,row in anno_clean.query("Event!='Session Start' and Event!='Session Stop'").iterrows():
            if row.Event=='Start':
                stack[row.Activity] = row.UnixTime_s
            else:
                if row.Activity in stack:
                    intervals.append((stack.pop(row.Activity), row.UnixTime_s, row.Activity))
        print("Intervals:", intervals)

        # Audio waveform and mel
        waveform, audio_start = rebuild_waveform(audio_df)
        print("Audio start:", audio_start)
        audio_examples = generate_mel_chunks(
            waveform, sr=16000, lower_edge_hertz=10,
            upper_edge_hertz=8000, chunk_secs=3600
        )
        print("Mel shape:", audio_examples.shape)

        # Timestamp array for mel frames
        hop = params.STFT_HOP_LENGTH_SECONDS
        win = params.STFT_WINDOW_LENGTH_SECONDS
        n_mels = audio_examples.shape[0]
        audio_ts= audio_start + np.arange(n_mels)*hop + win

        # Mel frames per example
        ex_len = int(params.EXAMPLE_WINDOW_SECONDS / hop)

        w_a, w_i, w_l = [], [], []

        # 올바른 시간 추적 방식으로 수정
        for i, (imu_window, time_window) in enumerate(zip(imu_frames, time_frames)):
            # 시간 윈도우에서 시작과 끝 시간 추출
            s_t = time_window[0, 0]     # 첫 번째 샘플의 시간
            e_t = time_window[-1, 0]    # 마지막 샘플의 시간

            s_idx = np.searchsorted(audio_ts, s_t)
            e_idx = s_idx + ex_len
            if e_idx > n_mels:
                continue

            seg = audio_examples[s_idx:e_idx]
            if seg.shape[0] < ex_len:
                seg = np.vstack([seg, np.zeros((ex_len-seg.shape[0], seg.shape[1]))])

            # 활동 라벨링
            ov = {act:0.0 for act in class_list}
            for st,et,act in intervals:
                ov[act] += max(0, min(e_t,et)-max(s_t,st))
            covered = sum(ov.values())
            ov['Other'] += max(0, (e_t-s_t)-covered)
            lbl = max(ov, key=ov.get)

            w_a.append(seg)
            w_i.append(imu_window)  # 이미 (100, 9) 형태
            w_l.append(lbl)

        X_audio = np.stack(w_a)  # (N, ex_len, 64)
        X_imu   = np.stack(w_i)  # (N, 100, 9) - 논문에 맞는 형태
        Y_lab   = np.array(w_l)

        print("Final shapes → audio:", X_audio.shape,
              "imu:", X_imu.shape, "labels:", Y_lab.shape)

        # Save to pickle
        out = SAVE_PATH / participant
        out.mkdir(parents=True, exist_ok=True)
        with open(out / f"{participant}_preprocessing.pkl", 'wb') as f:
            pkl.dump({'IMU':X_imu, 'Audio':X_audio, 'Activity':Y_lab}, f, protocol=4)